# 05 · Story Builder Demo

Two demos in one notebook.

**Section 1 — Cross-namespace comparison:** The same query goes to all three namespaces simultaneously. Results are shown side-by-side so you can see exactly how each embedding approach retrieves different pages for the same search.

**Section 2 — Narrative assembler:** Chain four queries (setup → conflict → climax → resolution) to stitch pages from multiple different comics into a brand-new story arc.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import voyageai
from dotenv import load_dotenv
from PIL import Image
from pinecone import Pinecone

load_dotenv()

TRANSCRIPTS_DIR = Path("data/transcripts")
INDEX_NAME      = "comics-multimodal"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
vo = voyageai.Client()

info = pc.preview.indexes.describe(name=INDEX_NAME)
index = pc.index(host=info.host)
print(f"Connected to {INDEX_NAME}")

In [ ]:
# Helper: embed query text into Voyage space (for ns-image-clip)
def voyage_query_vector(text: str) -> list[float]:
    result = vo.multimodal_embed([[text]], model="voyage-multimodal-3", input_type="query")
    return result.embeddings[0]


# Helper: search a namespace, return matches
def search_namespace(namespace: str, query: str, top_k: int = 5) -> list:
    if namespace == "ns-image-clip":
        # Voyage multimodal vector — text and image in the same space
        qvec = voyage_query_vector(query)
        score_by = [{"type": "dense_vector", "field": "embedding", "query": qvec}]
    else:
        # llama-text-embed-v2 via integrated inference
        score_by = [{"type": "dense_vector", "field": "embedding", "query": query}]

    resp = index.documents.search(
        namespace=namespace,
        top_k=top_k,
        score_by=score_by,
        include_fields=["page_id", "comic_title", "page_num", "file_path", "llm_caption"],
    )
    return resp.matches

---
## Section 1 — Cross-namespace comparison

Same query, three namespaces, side-by-side.

In [ ]:
def compare_namespaces(query: str, top_k: int = 3):
    namespaces = [
        ("ns-text-caption", "LLM Caption\n(dense)"),
        ("ns-image-clip",   "Voyage Image\n(visual)"),
        ("ns-hybrid",       "Hybrid\n(dense+sparse+FTS)"),
    ]

    all_results = {}
    for ns, _ in namespaces:
        all_results[ns] = search_namespace(ns, query, top_k=top_k)

    fig, axes = plt.subplots(top_k, len(namespaces), figsize=(14, top_k * 4))
    fig.suptitle(f'Query: "{query}"', fontsize=13, fontweight="bold", y=1.01)

    for col, (ns, ns_label) in enumerate(namespaces):
        axes[0][col].set_title(ns_label, fontsize=11, pad=8)
        for row, match in enumerate(all_results[ns]):
            ax = axes[row][col]
            fp = getattr(match, "file_path", None)
            if fp and Path(fp).exists():
                img = Image.open(fp).convert("RGB")
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, "no image", ha="center", va="center")
            comic = getattr(match, "comic_title", "") or ""
            page  = getattr(match, "page_num", "") or ""
            ax.set_xlabel(f"{comic}\np{page}  score={match._score:.3f}", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
compare_namespaces("detective discovers a clue")

In [ ]:
compare_namespaces("KAPOW punch fight action")

In [ ]:
compare_namespaces("mysterious figure in shadows")

---
## Section 2 — Narrative Assembler

Chain four queries — one per story beat — and assemble the top matching page from each into a 2×2 comic grid.

Pages are deduplicated: if two beats return the same page, we take the next best match.

In [ ]:
def assemble_story(story_beats: list[tuple[str, str]], namespace: str = "ns-hybrid"):
    """
    story_beats: list of (beat_label, query_text)
    Fetches top pages per beat (with dedup) and renders a grid.
    """
    used_ids = set()
    selected = []  # list of (label, match)

    for label, query in story_beats:
        matches = search_namespace(namespace, query, top_k=10)
        for m in matches:
            if m._id not in used_ids:
                used_ids.add(m._id)
                selected.append((label, m))
                break

    n = len(selected)
    cols = 2
    rows = (n + 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 5))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (label, match) in enumerate(selected):
        ax = axes[i]
        fp = getattr(match, "file_path", None)
        if fp and Path(fp).exists():
            ax.imshow(Image.open(fp).convert("RGB"))
        comic   = getattr(match, "comic_title", "") or ""
        page    = getattr(match, "page_num", "") or ""
        caption = getattr(match, "llm_caption", "") or ""
        ax.set_title(f"{label}", fontsize=11, fontweight="bold")
        ax.set_xlabel(f"{comic}  ·  p{page}\n{caption[:80]}…", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    # Hide unused axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Assembled Story Arc", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
assemble_story([
    ("1. Setup",      "detective arrives at a crime scene"),
    ("2. Conflict",   "villain threatens the hero"),
    ("3. Climax",     "dramatic fight or physical confrontation"),
    ("4. Resolution", "hero wins and villain is defeated"),
])

In [ ]:
# Try your own story arc:
assemble_story([
    ("1. Setup",      "mysterious stranger arrives in town"),
    ("2. Tension",    "secret revealed in a dimly lit room"),
    ("3. Climax",     "desperate chase through the streets"),
    ("4. Twist",      "unexpected alliance between enemies"),
])